In [0]:
# Creating a common schema for all layers
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.retail_analytics")

DataFrame[]

In [0]:
from pyspark.sql.functions import current_timestamp, lit, to_date

# 1. Configuration
volume_path = "/Volumes/workspace/default/landing_zone/"
target_schema = "workspace.retail_analytics"
source_system_name = "Legacy_Retail_CSV"

def ingest_to_bronze(file_name, table_name):
    # Reading raw data 
    df_raw = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "true")
              .load(f"{volume_path}{file_name}"))
    
    # Adding mandatory metadata
    df_bronze = (df_raw
                 .withColumn("ingestion_timestamp", current_timestamp())
                 .withColumn("source_system", lit(source_system_name))
                 .withColumn("ingestion_date", to_date("ingestion_timestamp")))

    # Saving  as Delta Table in the common location 
    (df_bronze.write
     .format("delta")
     .mode("overwrite") 
     .partitionBy("ingestion_date")
     .saveAsTable(f"{target_schema}.bronze_{table_name}"))
    
    print(f"Bronze table created: {target_schema}.bronze_{table_name}")

# Executing ingestion 
ingest_to_bronze("sales_transactions.csv", "sales")
ingest_to_bronze("products.csv", "products")
ingest_to_bronze("stores.csv", "stores")

Bronze table created: workspace.retail_analytics.bronze_sales
Bronze table created: workspace.retail_analytics.bronze_products
Bronze table created: workspace.retail_analytics.bronze_stores


In [0]:
from pyspark.sql.functions import col, when, current_timestamp, lit, to_utc_timestamp, expr
from delta.tables import DeltaTable

# 1. Configuration & Constants
target_schema = "workspace.retail_analytics"
FIXED_EXCHANGE_RATE = 1.1  # EUR to USD 

# 1. Incremental sales processing
# Purpose: cleanse, calibrate, and incrementally merge into Silver 

# Getting the last processed timestamp for Watermarking 
try:
    last_timestamp = spark.sql(f"SELECT max(ingestion_timestamp) FROM {target_schema}.silver_sales").collect()[0][0]
except:
    last_timestamp = '1900-01-01 00:00:00'

# Reading only new data from Bronze (Incremental Ingestion) 
bronze_sales_new = spark.read.table(f"{target_schema}.bronze_sales") \
    .filter(col("ingestion_timestamp") > last_timestamp)

# Data Quality Rules: Identify "Dirty" records for Quarantine 
is_invalid = (col("store_id").isNull()) | (col("product_id").isNull()) | (col("quantity") <= 0)

# Divert invalid records to Quarantine table 
quarantine_df = bronze_sales_new.filter(is_invalid) \
    .withColumn("rejection_reason", 
                when(col("store_id").isNull(), "Null Store ID")
                .when(col("product_id").isNull(), "Null Product ID")
                .otherwise("Quantity <= 0"))

if quarantine_df.count() > 0:
    quarantine_df.write.format("delta").mode("append").saveAsTable(f"{target_schema}.silver_quarantine")

# Process Valid Records: Calibration & Standardization 
valid_sales_df = bronze_sales_new.filter(~is_invalid) \
    .withColumn("transaction_timestamp", to_utc_timestamp(col("transaction_timestamp"), "UTC")) \
    .withColumn("total_amount_usd", 
                when(col("currency") == "EUR", col("total_amount") * FIXED_EXCHANGE_RATE)
                .otherwise(col("total_amount"))) \
    .withColumn("expected_total", (col("quantity") * col("unit_price")) - col("discount")) \
    .withColumn("is_calibrated", when(col("total_amount") != col("expected_total"), True).otherwise(False)) \
    .withColumn("total_amount", col("expected_total")) \
    .drop("expected_total", "ingestion_date") \
    .dropDuplicates(["transaction_id"]) # Removing duplicates 

# Incremental MERGE (UPSERT) into Silver 
if not spark.catalog.tableExists(f"{target_schema}.silver_sales"):
    valid_sales_df.write.format("delta").saveAsTable(f"{target_schema}.silver_sales")
else:
    silver_table = DeltaTable.forName(spark, f"{target_schema}.silver_sales")
    silver_table.alias("target").merge(
        valid_sales_df.alias("source"),
        "target.transaction_id = source.transaction_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# 2. PRODUCT MASTER (SCD TYPE 2)
# Purpose: track price changes over time 
bronze_products = spark.read.table(f"{target_schema}.bronze_products")

# In a full SCD2, we would join and expire old rows. 
# For this simulation, we ensure only unique product versions exist.
silver_products = bronze_products \
    .withColumn("is_current", lit(True)) \
    .withColumn("effective_date", col("ingestion_timestamp")) \
    .dropDuplicates(["product_id", "standard_price"])

silver_products.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true").saveAsTable(f"{target_schema}.silver_products")

#3. STORE REFERENCE
spark.read.table(f"{target_schema}.bronze_stores").dropDuplicates(["store_id"]) \
    .write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.silver_stores")

print("Silver layer incremental update complete.") 

Silver layer incremental update complete.


In [0]:
from pyspark.sql.functions import sum, count, col, date_format, desc, rank
from pyspark.sql.window import Window

# 1. Configuration
target_schema = "workspace.retail_analytics"

# Loading Silver Tables
silver_sales = spark.read.table(f"{target_schema}.silver_sales")
silver_products = spark.read.table(f"{target_schema}.silver_products")
silver_stores = spark.read.table(f"{target_schema}.silver_stores")

#  2. GOLD TABLE 1: Executive Sales Overview 
# Purpose: total revenue, trends, and top regions 
# Daily Sales Summary and Monthly Revenue by Region included in GOLD TABLE 1
exec_sales_df = (silver_sales
    .join(silver_stores, "store_id")
    .groupBy(
        date_format("transaction_timestamp", "yyyy-MM-dd").alias("sales_date"),
        "region", 
        "country"
    )
    .agg(
        sum("total_amount").alias("total_revenue_usd"),
        count("transaction_id").alias("transaction_count")
    )
)

exec_sales_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.gold_exec_sales_overview")

#3. GOLD TABLE 2: Product Performance 
# Purpose: top products and category-wise sales trends
product_perf_df = (silver_sales
    .join(silver_products, "product_id")
    .groupBy(
        date_format("transaction_timestamp", "yyyy-MM").alias("sales_month"),
        "category",
        "product_name"
    )
    .agg(
        sum("total_amount").alias("monthly_revenue"),
        sum("quantity").alias("units_sold")
    )
)

# Adding a rank to ensure Power BI shows "Top Products"
windowSpec = Window.partitionBy("sales_month").orderBy(desc("monthly_revenue"))
product_perf_df = product_perf_df.withColumn("product_rank", rank().over(windowSpec))

product_perf_df.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.gold_product_performance")

print("Gold tables created and optimized for Power BI.")

Gold tables created and optimized for Power BI.
